# F1 Data Loader
Downloads F1 telemetry data from FastF1 API. Run once per year in incremental way due to API usage limits (100 req/h for IP).

### Workflow
For each year/execution:
1. Set `YEAR_TO_LOAD = <year>`
2. Execute all cells
3. Download file `f1_dataset_combined.pkl`
4. **Disconnect runtime** (to avoid IP ban)
5. Reload pickle and repeat for year++

### Output
| File | Descrizione |
|------|-------------|
| `f1_dataset_combined.pkl` | Incremental dataset (growing) |
| `Cache/` | Local cache FastF1 |

---
## 1. Setup

In [1]:
# warnings is a built-in Python module for controlling warning messages
import warnings
# filterwarnings() controls which warnings are shown
# 'ignore' means hide all warning messages in output
warnings.filterwarnings('ignore')


import os
# Crea directory cache
os.makedirs('Cache', exist_ok=True) # exist_ok=True means don't raise an error if folder already exists


import pandas as pd


# fastf1 is an open-source library that downloads F1 telemetry data
# it connects to the official F1 timing API
!pip install fastf1 -q
import fastf1
# enable_cache() tells fastf1 to save downloaded data locally
fastf1.Cache.enable_cache('Cache')


# logging is a built-in module for controlling log messages (info, debug, errors)
import logging
# getLogger() gets the logger object for a specific library
# setLevel() sets the minimum severity level to show
logging.getLogger('fastf1').setLevel(logging.ERROR)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 123.0/123.0 kB 11.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 138.0/138.0 kB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.4/61.4 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 96.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.7/70.7 kB 6.7 MB/s eta 0:00:00


## 2. Loader

In [2]:
class F1DataLoaderIncremental:

    def __init__(self, cache_dir='Cache', output_file='f1_dataset_combined.pkl'):
        self.cache_dir = cache_dir
        self.output_file = output_file
        self.df = None
        fastf1.Cache.enable_cache(cache_dir)



    def load_existing_data(self):
        if os.path.exists(self.output_file):
            print(f"Found existing data: {self.output_file}")

            # pd.read_pickle() loads a DataFrame from a pickle file
            # pickle is a Python format that preserves all data types
            # it's faster than CSV and keeps dtypes intact
            existing = pd.read_pickle(self.output_file)

            years_present = sorted(existing['Year'].unique())
            print(f"   Years already loaded: {years_present}")

            return existing

        else:
            print("No existing data - first execution")

            return None



    def _extract_safety_car_laps(self, session):

        # dictionary stores key-value pairs like {15: 'SC', 16: 'SC'}
        sc_laps = {}

        try:

            # session.race_control_messages is a DataFrame of official race messages
            # contains messages like "SAFETY CAR DEPLOYED", "GREEN LIGHT" etc.
            rcm = session.race_control_messages
            if rcm is None or rcm.empty:
                return sc_laps

            # current_status tracks if we're currently under SC or VSC
            # None means no safety car active
            current_status = None

            for _, msg in rcm.iterrows():
                message = str(msg.get('Message', '')).upper()
                lap = msg.get('Lap', None)

                if 'SAFETY CAR DEPLOYED' in message or 'SAFETY CAR IN THIS LAP' in message:
                    current_status = 'SC'
                    if lap is not None:
                        sc_laps[int(lap)] = 'SC'

                elif 'VIRTUAL SAFETY CAR DEPLOYED' in message or 'VSC DEPLOYED' in message:
                    current_status = 'VSC'
                    if lap is not None:
                        sc_laps[int(lap)] = 'VSC'

                elif 'SAFETY CAR IN' in message and 'THIS LAP' not in message:
                    if lap is not None and current_status == 'SC':
                        sc_laps[int(lap)] = 'SC'
                    current_status = None

                elif 'VSC ENDING' in message or 'GREEN LIGHT' in message:
                    if lap is not None and current_status == 'VSC':
                        sc_laps[int(lap)] = 'VSC'
                    current_status = None

                # if still under caution and haven't recorded this lap yet
                elif current_status is not None and lap is not None:
                    if int(lap) not in sc_laps:
                        sc_laps[int(lap)] = current_status

        except Exception as e:

            print(f"      Warning: Error extracting SC/VSC: {e}")

        return sc_laps



    def load_single_year(self, year):

        # will store DataFrames from each race
        all_laps = []
        errors = []

        # fastf1.get_event_schedule() downloads the F1 calendar for a year
        # returns DataFrame with race names, dates, locations
        schedule = fastf1.get_event_schedule(year)

        # schedule['EventFormat'] selects the EventFormat column
        # this filters out testing sessions (pre-season testing)
        # we only want actual race weekends
        races = schedule[schedule['EventFormat'] != 'testing']
        n_rounds = len(races)
        print(f"Races to load: {n_rounds}\n")

        # counter for successfully loaded races
        successful = 0
        for idx in range(n_rounds):

            race = races.iloc[idx]
            round_num = race['RoundNumber']
            race_name = race['EventName']
            location = race['Location']
            print(f"Loading race {idx+1}/{n_rounds}: {race_name}...")

            try:

                # fastf1.get_session() gets a specific session
                # round_num = which race in the calendar (1, 2, 3, ...)
                # 'R' = Race session
                # could also be 'Q' for Qualifying, 'FP1' for Practice 1, etc.
                session = fastf1.get_session(year, round_num, 'R')

                # session.load() downloads the actual data from the API
                # laps=True: download lap timing data (lap times, positions, compounds)
                # telemetry=False: skip detailed car telemetry (API usage)
                # weather=True: download weather information (temperature, rain)
                # messages=True: download race control messages (for SC detection)
                session.load(laps=True, telemetry=False, weather=True, messages=True)

                # .copy() creates an independent copy of the DataFrame
                laps = session.laps.copy()
                laps = laps[laps['LapTime'].notna()]

                # 'in' checks if string exists in list
                # laps.columns is a list of all column names in DataFrame
                # some sessions might not have IsAccurate column
                if 'IsAccurate' in laps.columns:
                    laps = laps[(laps['IsAccurate'] == True)]

                # len(laps) returns number of rows
                # > 0 checks if we have any data
                if len(laps) > 0:

                    laps['Year'] = year
                    laps['Round'] = round_num
                    laps['RaceName'] = race_name
                    laps['Circuit'] = location
                    laps['Country'] = race['Country']
                    laps['EventDate'] = race['EventDate']

                    # hasattr() checks if an object has a specific attribute/property
                    # session.weather_data might not exist for all sessions
                    if hasattr(session, 'weather_data') and session.weather_data is not None:
                        weather = session.weather_data
                        if not weather.empty:
                            # Convert weather Time to seconds for matching
                            weather = weather.copy()
                            weather['WeatherTimeSec'] = weather['Time'].dt.total_seconds()
                            
                            # Convert lap start time to seconds
                            laps['LapStartTimeSec'] = laps['LapStartTime'].dt.total_seconds()
                            
                            # Sort laps by time for merge_asof
                            laps = laps.sort_values('LapStartTimeSec').reset_index(drop=True)
                            
                            # Merge weather to each lap based on closest timestamp
                            # direction='backward' uses most recent weather reading before lap
                            laps = pd.merge_asof(
                                laps,
                                weather[['WeatherTimeSec', 'AirTemp', 'TrackTemp', 'Humidity', 'Pressure', 'WindSpeed', 'Rainfall']],
                                left_on='LapStartTimeSec',
                                right_on='WeatherTimeSec',
                                direction='backward'
                            )
                            
                            # Drop temporary columns
                            laps = laps.drop(columns=['LapStartTimeSec', 'WeatherTimeSec'], errors='ignore')

                    sc_laps = self._extract_safety_car_laps(session)
                    laps['SafetyCarStatus'] = laps['LapNumber'].apply(
                        lambda x: sc_laps.get(int(x), 'NONE') if pd.notna(x) else 'NONE'
                    )

                    laps = self._add_gap_features(laps)
                    all_laps.append(laps)
                    successful += 1
                    print(f"   Loaded {len(laps)} laps")

                else:
                    print(f"   No valid laps found")
                    errors.append(f"{race_name}: No laps")

            except Exception as e:
                print(f"   Error: {e}")
                errors.append(f"{race_name}: {str(e)}")


        if all_laps:

            # ignore_index=True resets the row index to 0, 1, 2, ...
            # without this, you'd have duplicate indices from each DataFrame
            df_year = pd.concat(all_laps, ignore_index=True)

            print(f"\n{'='*60}")
            print(f"YEAR {year} SUMMARY")
            print(f"{'='*60}")


            print(f"Races loaded:   {len(df_year.groupby('Round'))}")
            print(f"Total laps:     {len(df_year):,}")
            print(f"Drivers:        {df_year['Driver'].nunique()}")
            print(f"Teams:          {df_year['Team'].nunique()}")
            print(f"Circuits:       {df_year['Circuit'].nunique()}")


            # if errors list is not empty, print them
            if errors:
                print(f"\nErrors ({len(errors)}):")
                for err in errors:
                    print(f"   - {err}")


            return df_year

        else:
            print(f"No data loaded for {year}")
            return None



    def _add_gap_features(self, laps):

        laps = laps.copy()
        laps = laps.sort_values(['LapNumber', 'Position']).reset_index(drop=True)

        gap_leader = []
        for lap_num in laps['LapNumber'].unique():


            lap_data = laps[laps['LapNumber'] == lap_num]
            if len(lap_data) > 0 and 'Time' in lap_data.columns:

                leader_time = lap_data.iloc[0]['Time']

                # subtract leader's time from everyone's time
                # lap_data['Time'] - leader_time does element-wise subtraction
                # result is timedelta (time difference)
                # .dt.total_seconds() converts timedelta to float seconds
                gaps = (lap_data['Time'] - leader_time).dt.total_seconds()

                # .tolist() converts pandas Series to Python list
                # .extend() adds ALL items from one list to another
                gap_leader.extend(gaps.tolist())

            else:
                # no time data available for this lap
                # [0.0] * n creates list of n zeros: [0.0, 0.0, 0.0, ...]
                # len(lap_data) is number of cars on this lap
                gap_leader.extend([0.0] * len(lap_data))

        laps['GapToLeader'] = gap_leader

        if 'Stint' in laps.columns:
            laps['StintProgress'] = laps.groupby(['Driver', 'Stint']).cumcount() + 1

        return laps



    def combine_and_save(self, new_data, existing_data=None):

        if existing_data is not None:
            combined = pd.concat([existing_data, new_data], ignore_index=True)

            # a row is duplicate if Year, Round, Driver, LapNumber all match
            # keep='first' keeps the first occurrence, removes later ones
            combined = combined.drop_duplicates(
                subset=['Year', 'Round', 'Driver', 'LapNumber'],
                keep='first'
            )

        else:
            # no existing data, just use the new data
            combined = new_data

        # .to_pickle() saves DataFrame to a pickle file
        # pickle is Python's serialization format
        # it preserves all data types (datetime, float, int, etc.)
        # faster to load than CSV and keeps types intact
        combined.to_pickle(self.output_file)

        print(f"\nSAVED: {self.output_file}")
        return combined



    def load_year_incremental(self, year):

        print("\n")
        print("=" * 60)
        print(f"Year to load: {year}")
        print(f"Output file: {self.output_file}")

        # call our method to load any existing data
        existing = self.load_existing_data()
        if existing is not None and year in existing['Year'].values:

            print(f"Year {year} already present!")
            return existing
        print("=" * 60)

        # download the year's data
        new_data = self.load_single_year(year)
        if new_data is None:

            print(f"Unable to load year {year}")
            return existing if existing is not None else None

        # combine new data with existing and save to file
        combined = self.combine_and_save(new_data, existing)
        return combined

---
## 3. Configuration

⚠️ **Change `YEAR_TO_LOAD`!**

In [3]:
YEAR_TO_LOAD = 2025

## 4. Execution

In [4]:
loader = F1DataLoaderIncremental(
    cache_dir='Cache',
    output_file='f1_dataset_combined.pkl'
)

df_f1 = loader.load_year_incremental(
    year=YEAR_TO_LOAD,
)



Year to load: 2025
Output file: f1_dataset_combined.pkl
Found existing data: f1_dataset_combined.pkl
   Years already loaded: [np.int64(2022), np.int64(2023), np.int64(2024)]
Races to load: 24

Loading race 1/24: Australian Grand Prix...
   Loaded 568 laps
Loading race 2/24: Chinese Grand Prix...
   Loaded 995 laps
Loading race 3/24: Japanese Grand Prix...
   Loaded 997 laps
Loading race 4/24: Bahrain Grand Prix...
   Loaded 952 laps
Loading race 5/24: Saudi Arabian Grand Prix...
   Loaded 810 laps
Loading race 6/24: Miami Grand Prix...
   Loaded 861 laps
Loading race 7/24: Emilia Romagna Grand Prix...
   Loaded 956 laps
Loading race 8/24: Monaco Grand Prix...
   Loaded 1271 laps
Loading race 9/24: Spanish Grand Prix...
   Loaded 990 laps
Loading race 10/24: Canadian Grand Prix...
   Loaded 1197 laps
Loading race 11/24: Austrian Grand Prix...
   Loaded 1010 laps
Loading race 12/24: British Grand Prix...
   Loaded 496 laps
Loading race 13/24: Belgian Grand Prix...
   Loaded 747 laps
L